# **Day 55: Generators**

Yesterday, making our own iterator needed a whole class with `__iter__` and `__next__`. A **generator** does the same job in a much **simpler** way, using the `yield` keyword.

**How does yield work?**
- `return` gives a value and **ends** the function.
- `yield` gives a value and **pauses** the function. The next time we ask for a value, it **continues** from where it stopped.

**Simple example:** A generator is like a **tea seller on a train**. He does not bring 100 cups at once. He gives you one cup, and when you ask again, he makes the next one.

In [1]:
def countdown(n):
    print("Starting")
    while n > 0:
        yield n
        n -= 1
    print("Done")

g = countdown(3)
print(g)                        # A generator object
print(next(g))
print(next(g))
print(next(g))
print(next(g, "Finished"))      # A default value instead of StopIteration

<generator object countdown at 0x000001E29BDFAB50>
Starting
3
2
1
Done
Finished


In [2]:
for x in countdown(4):
    print(x, end=" ")

Starting
4 3 2 1 Done


## **Generator vs the Iterator Class From Yesterday**

In [3]:
def fibonacci():
    a, b = 0, 1
    while True:              # An infinite generator (it never ends by itself)
        yield a
        a, b = b, a + b

from itertools import islice
print(list(islice(fibonacci(), 15)))     # Take only the first 15 values

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]


Only 4 lines of code, compared to a full class.

## **Generators Save Memory**

In [4]:
import sys

def squares_list(n):
    return [i * i for i in range(n)]

def squares_gen(n):
    for i in range(n):
        yield i * i

print(sys.getsizeof(squares_list(1_000_000)), "bytes for the list")
print(sys.getsizeof(squares_gen(1_000_000)), "bytes for the generator")
print("Sum:", sum(squares_gen(1_000_000)))

8448728 bytes for the list
208 bytes for the generator
Sum: 333332833333500000


## **Example from Remote Sensing: A Tile Generator**
Let us rewrite yesterday's `TileIterator` class as a generator. It becomes much shorter.

In [5]:
def tiles(height, width, tile_size):
    for r0 in range(0, height, tile_size):
        for c0 in range(0, width, tile_size):
            yield r0, min(r0 + tile_size, height), c0, min(c0 + tile_size, width)

print(len(list(tiles(1000, 1500, 512))), "tiles")
print(list(tiles(1000, 1500, 512))[:3])

6 tiles
[(0, 512, 0, 512), (0, 512, 512, 1024), (0, 512, 1024, 1500)]


## **Pipeline of Generators**
We can connect generators like **pipes**: each one does one small job and passes the result to the next. Even for millions of rows, only **one row at a time** is in memory.

**Example:** A GPS tracker in a field survey writes a log file. We want to:
1. Read the lines
2. Skip bad lines
3. Keep only the points where the speed is below 5 km/h (the places where the team stopped to survey)

In [6]:
gps_log = """
2024-01-15 09:01:10,21.9452,88.8931,4.1
2024-01-15 09:05:40,21.9470,88.8960,18.5
bad line here
2024-01-15 09:12:02,21.9610,88.9105,2.3
2024-01-15 09:20:15,21.9700,88.9050,22.0
2024-01-15 09:31:45,21.9877,88.8702,0.8
2024-01-15 09:40:00,21.9901,88.8801,3.9
""".strip().splitlines()

def read_lines(lines):
    for line in lines:
        yield line.strip()

def parse(lines):
    for line in lines:
        parts = line.split(",")
        if len(parts) != 4:
            continue                       # Skip bad lines
        time, lat, lon, speed = parts
        yield {"time": time[11:], "lat": float(lat), "lon": float(lon), "speed": float(speed)}

def stops(points, max_speed=5):
    for p in points:
        if p["speed"] < max_speed:
            yield p

pipeline = stops(parse(read_lines(gps_log)))
for p in pipeline:
    print(p)

{'time': '09:01:10', 'lat': 21.9452, 'lon': 88.8931, 'speed': 4.1}
{'time': '09:12:02', 'lat': 21.961, 'lon': 88.9105, 'speed': 2.3}
{'time': '09:31:45', 'lat': 21.9877, 'lon': 88.8702, 'speed': 0.8}
{'time': '09:40:00', 'lat': 21.9901, 'lon': 88.8801, 'speed': 3.9}


Let us understand the code above:
- `read_lines` gives one line at a time.
- `parse` turns each line into a dictionary and skips the bad line.
- `stops` keeps only the slow points.
- Each generator **pulls** one item from the one before it. No full list is ever made.

## **yield from**
`yield from` passes all the values from another generator.

In [7]:
def north_bengal():
    yield from ["Siliguri", "Jalpaiguri", "Cooch Behar"]

def south_bengal():
    yield from ["Kolkata", "Howrah", "Midnapore"]

def all_places():
    yield from north_bengal()
    yield from south_bengal()
    yield "Sundarbans"

print(list(all_places()))

['Siliguri', 'Jalpaiguri', 'Cooch Behar', 'Kolkata', 'Howrah', 'Midnapore', 'Sundarbans']


## **Practice Problem: Split a List into Batches**
This is very useful when sending data to a website in small parts.

In [8]:
def batches(items, size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

districts = ["Kolkata", "Howrah", "Hooghly", "Nadia", "Bankura", "Purulia", "Birbhum",
             "Malda", "Darjeeling", "Jalpaiguri", "Kalimpong"]

for i, b in enumerate(batches(districts, 4), 1):
    print(i, b)

1 ['Kolkata', 'Howrah', 'Hooghly', 'Nadia']
2 ['Bankura', 'Purulia', 'Birbhum', 'Malda']
3 ['Darjeeling', 'Jalpaiguri', 'Kalimpong']


In [9]:
# Quick question
# Guess the output of this code cell
# def g():
#     yield 1
#     yield 2
# x = g()
# print(next(x) + next(x))